# IOAI — 2026 Round2 Mix Functions (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
os.makedirs('data', exist_ok=True)
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-round2-mix-functions'
for f in ['train.csv','train_params.csv','test.csv']:
    if not os.path.exists(f'data/{f}'): os.system(f'wget -q {BASE}/{f} -O data/{f}')
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Mixed Functions — USAAIO 2026 Round 2 (Day 2, Problem 5)

각 **그룹**은 400개의 관측점 `(x, y)` 로 이루어지고, 이 점들은 네 함수에서 각각 100개씩(섞여서) 나온다:

- 지수: `y = a0·exp(a1·x) + a2`  · 로그: `y = b0·ln(x) + b1`
- 삼각: `y = c0·sin(c1·x + c2) + c3`  · 선형: `y = d0·x + d1`

과제: 각 그룹의 400점만 보고 **11개 파라미터** `a0,a1,a2,b0,b1,c0,c1,c2,c3,d0,d1` 을 추정한다.
`data/train.csv`(`group_id,x,y`)와 `data/train_params.csv`(참값)로 방법을 검증하고, `data/test.csv`(파라미터 숨김)의
각 그룹에 대해 예측해 `submission.csv`(`group_id,a_0..d_1`)를 만든다.

**채점**: 예측 파라미터로 만든 4함수 중 각 점에 **가장 잘 맞는 함수**의 잔차 RMSE(↓). 참값이면 ≈0.12(노이즈 바닥).


In [ ]:
import numpy as np, pandas as pd
from scipy.optimize import curve_fit

test = pd.read_csv("data/test.csv")     # group_id, x, y
PCOLS = ["a_0","a_1","a_2","b_0","b_1","c_0","c_1","c_2","c_3","d_0","d_1"]

# 베이스라인: 각 그룹에 '선형' 하나만 맞추고 나머지 파라미터는 0 (부분점수만)
rows = []
for g, gg in test.groupby("group_id"):
    d0, d1 = np.polyfit(gg["x"], gg["y"], 1)
    p = {"a_0":0,"a_1":0,"a_2":0,"b_0":0,"b_1":0,"c_0":0,"c_1":0,"c_2":0,"c_3":0,"d_0":d0,"d_1":d1}
    rows.append({"group_id": g, **p})
pd.DataFrame(rows)[["group_id"]+PCOLS].to_csv("submission.csv", index=False)
print("saved submission.csv", len(rows), "groups")

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)